# 02 — Custom Nozzle Design

This notebook demonstrates how to design a custom magnetic nozzle:
1. Define coil geometry programmatically
2. Compute the B-field and inspect the magnetic topology
3. Use analytical pre-screening to estimate performance
4. Run a dry-run simulation with the custom configuration
5. Compare against a preset reference case


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from helicon.fields.biot_savart import Coil, Grid, compute_bfield
from helicon.optimize.analytical import screen_geometry

## 1. Define a custom two-coil converging-diverging nozzle

In [ ]:
# Throat coil (high-current, small radius)
coil_throat = Coil(z=0.0, r=0.10, I=60_000.0)
# Diverging coil (lower current, larger radius, downstream)
coil_div = Coil(z=0.08, r=0.18, I=15_000.0)

coils = [coil_throat, coil_div]

# Compute the field on a 2D axisymmetric grid
grid = Grid(z_min=-0.3, z_max=1.5, r_max=0.5, nz=256, nr=128)
bfield = compute_bfield(coils, grid)
print(f"B_z max at throat: {bfield.Bz.max():.3f} T")
print(f"B_z at r=0, z=1.5: {bfield.Bz[bfield.z.argmax(), 0]:.4f} T")

## 2. Visualize the magnetic field

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# B_z contour
Z, R = np.meshgrid(bfield.z, bfield.r)
pcm = axes[0].pcolormesh(Z, R, bfield.Bz.T, cmap='RdBu_r', shading='auto')
plt.colorbar(pcm, ax=axes[0], label='$B_z$ (T)')
axes[0].set_xlabel('z (m)')
axes[0].set_ylabel('r (m)')
axes[0].set_title('$B_z$ — Axial Field Component')
# Mark coil positions
for coil in coils:
    axes[0].axvline(coil.z, color='k', ls='--', lw=0.8, alpha=0.5)
    axes[0].plot(coil.z, coil.r, 'ko', ms=6)

# On-axis profile
z_ax = bfield.z
Bz_ax = bfield.Bz[:, 0]
axes[1].plot(z_ax, Bz_ax, 'b-', lw=2)
axes[1].set_xlabel('z (m)')
axes[1].set_ylabel('$B_z$ on axis (T)')
axes[1].set_title('On-Axis $B_z$ Profile')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(0, color='r', ls=':', label='throat')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Analytical pre-screening

In [ ]:
result = screen_geometry(coils, z_min=-0.3, z_max=1.5)

print(f"Mirror ratio:       R_B = {result.mirror_ratio:.2f}")
print(f"Thrust efficiency:  η_T = {result.thrust_efficiency:.3f}")
print(f"Divergence angle:   θ   = {result.divergence_half_angle_deg:.1f}°")
print()
if result.mirror_ratio >= 3:
    print("✓ Good mirror ratio (≥ 3) — expect η_det > 0.80")
else:
    print("⚠ Low mirror ratio — consider increasing throat current or reducing exhaust field")

## 4. Build a custom SimConfig

In [ ]:
from helicon.config.parser import (
    CoilConfig, DomainConfig, NozzleConfig,
    PlasmaSourceConfig, ResolutionConfig, SimConfig,
)

config = SimConfig(
    nozzle=NozzleConfig(
        type="converging_diverging",
        coils=[
            CoilConfig(z=0.0, r=0.10, I=60_000.0),
            CoilConfig(z=0.08, r=0.18, I=15_000.0),
        ],
        domain=DomainConfig(z_min=-0.3, z_max=1.5, r_max=0.5),
        resolution=ResolutionConfig(nz=256, nr=128),
    ),
    plasma=PlasmaSourceConfig(
        species=["D+", "e-"],
        n0=1e19,
        T_i_eV=5000,
        T_e_eV=2000,
        v_injection_ms=150_000,
    ),
    timesteps=10000,
    output_dir="results/custom_nozzle",
)
print(config.model_dump_json(indent=2)[:500], "...")

## 5. Dry-run simulation (generates input files)

In [ ]:
from helicon.runner.launch import run_simulation

result = run_simulation(config, output_dir="results/custom_nozzle", dry_run=True)
print(f"Input file:  {result.input_file}")
print(f"B-field file: {result.bfield_file}")
print(f"Run dir:      {result.output_dir}")

## 6. Compare against DFD preset

In [ ]:
dfd_config = SimConfig.from_preset("dfd")
dfd_coils = [
    Coil(z=c.z, r=c.r, I=c.I)
    for c in dfd_config.nozzle.coils
]
dfd_result = screen_geometry(
    dfd_coils,
    z_min=dfd_config.nozzle.domain.z_min,
    z_max=dfd_config.nozzle.domain.z_max,
)

print("            Custom    DFD Preset")
print(f"Mirror ratio: {result.mirror_ratio:6.2f}    {dfd_result.mirror_ratio:6.2f}")
print(f"η_T:          {result.thrust_efficiency:6.3f}    {dfd_result.thrust_efficiency:6.3f}")
print(f"θ_div:        {result.divergence_half_angle_deg:5.1f}°    {dfd_result.divergence_half_angle_deg:5.1f}°")